# Notebook 09: Ray Serve

## Why This Matters

Training a model is half the battle. Serving it in production -- with low latency, high throughput, auto-scaling, and zero-downtime deployments -- is the other half. Ray Serve is a model serving framework built on Ray that handles batching, replica management, pipeline chaining, and integration with FastAPI. Unlike TorchServe (per-model only) or Triton (primarily GPU), Ray Serve handles heterogeneous deployments -- CPU preprocessors, GPU models, and complex routing logic all in one framework.


In [ ]:
%pip install -q 'ray[serve]' torch fastapi httpx starlette

In [ ]:
import ray
from ray import serve
import torch
import torch.nn as nn
import numpy as np
from starlette.requests import Request
import asyncio, time

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

# Note: serve.start() called when we first deploy
print(f"Ray Serve version: {serve.__version__}")
print("Ready.")

## 1. Basic Deployment

The core abstraction: `@serve.deployment`. A deployment is a class or function that handles HTTP requests. Ray Serve manages:
- **Replicas**: multiple copies of the deployment for parallelism
- **Scaling**: auto-scale replicas based on request queue length
- **Load balancing**: distribute requests across replicas


In [ ]:
# Basic deployment
@serve.deployment(num_replicas=2)  # 2 parallel replicas
class SimpleClassifier:
    def __init__(self):
        # Load model once at initialization
        self.model = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 5)
        )
        self.model.eval()
        print('Model loaded on replica')
    
    async def __call__(self, request: Request):
        # Handle HTTP POST request
        data = await request.json()
        features = torch.tensor(data['features'], dtype=torch.float32)
        
        with torch.no_grad():
            logits = self.model(features)
            probs = torch.softmax(logits, dim=-1)
            pred = logits.argmax(dim=-1)
        
        return {
            'prediction': pred.tolist(),
            'probabilities': probs.tolist()
        }

# Deploy
serve.start(detached=False)
handle = serve.run(SimpleClassifier.bind(), route_prefix='/classify')
print('Deployment ready!')

# Test with direct handle call (no HTTP)
import asyncio

async def test_deployment(handle):
    features = np.random.randn(4, 16).tolist()
    
    # Create a mock request
    from starlette.testclient import TestClient
    import json
    
    # Use handle.remote for direct invocation
    result = await handle.remote(features=features)
    return result

# Test via HTTP would require running requests.post('http://localhost:8000/classify')
print('Deployment active. In production, send HTTP POST to :8000/classify')
print('Request format: {"features": [[1.0, 2.0, ...], ...]}')

## 2. Batching for Throughput

Single-request inference wastes GPU utilization -- you pay the latency cost of a GPU launch for just one example. **Batching** groups multiple requests together before running inference.

Ray Serve's `@serve.batch` decorator automatically accumulates requests and processes them as a batch. Key parameters:
- `max_batch_size`: max examples per batch
- `batch_wait_timeout_s`: how long to wait for a full batch before processing a partial batch


In [ ]:
# Batching for throughput
@serve.deployment
class BatchedClassifier:
    def __init__(self):
        self.model = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 5)
        )
        self.model.eval()
    
    @serve.batch(max_batch_size=32, batch_wait_timeout_s=0.01)
    async def batch_handler(self, features_list):
        # features_list is a list of tensors from multiple requests
        batch = torch.stack([torch.tensor(f, dtype=torch.float32) for f in features_list])
        
        with torch.no_grad():
            logits = self.model(batch)
            preds = logits.argmax(dim=-1)
        
        # Return one result per input request
        return preds.tolist()
    
    async def __call__(self, request: Request):
        data = await request.json()
        result = await self.batch_handler(data['features'])
        return {'prediction': result}

print('Batched deployment pattern:')
print('  @serve.batch accumulates up to 32 requests into one batch')
print('  If batch not full after 10ms, process partial batch')
print('  GPU utilization: much higher than single-request serving')

# Latency vs throughput tradeoff
batch_sizes = [1, 4, 8, 16, 32, 64]
single_latency_ms = 5.0  # base latency per batch (GPU launch overhead)
per_example_ms = 0.1     # marginal cost per example

print('\nBatch size vs throughput analysis:')
print(f'{"Batch size":>12} | {"Latency (ms)":>12} | {"Throughput (req/s)":>18}')
print('-' * 50)
for bs in batch_sizes:
    latency = single_latency_ms + bs * per_example_ms
    throughput = bs / (latency / 1000)
    print(f'{bs:>12} | {latency:>12.1f} | {throughput:>18.0f}')

## 3. Deployment Pipelines and Multiplexing

Ray Serve supports chaining deployments into pipelines (preprocessor -> model -> postprocessor) and multiplexing multiple models behind one endpoint.


In [ ]:
# Pipeline: chain deployments
@serve.deployment
class Preprocessor:
    def __call__(self, text: str) -> list:
        # Simulate tokenization
        tokens = text.lower().split()[:16]
        # Pad/truncate to 16
        padded = tokens + ['[PAD]'] * (16 - len(tokens))
        # Fake embedding: hash to float
        embedding = [hash(t) % 100 / 100.0 for t in padded]
        return embedding

@serve.deployment
class TextClassifier:
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor  # handle to upstream deployment
        self.model = nn.Sequential(nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 3))
        self.model.eval()
    
    async def __call__(self, text: str) -> dict:
        # Call upstream preprocessor
        embedding = await self.preprocessor.remote(text)
        
        features = torch.tensor(embedding, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            logits = self.model(features)
            pred = logits.argmax(dim=-1).item()
        return {'text': text[:50], 'prediction': pred}

preprocessor_handle = Preprocessor.bind()
classifier_handle = TextClassifier.bind(preprocessor_handle)

print('Pipeline architecture:')
print('  Request -> Preprocessor -> TextClassifier -> Response')
print('  Each stage runs in its own Ray deployment (separate scaling)')

print('\nKey serving comparison:')
comparisons = [
    ('TorchServe', 'Single model, Java-based, mature', 'Simple PyTorch serving'),
    ('Triton', 'Multi-framework, GPU-optimized, C++', 'High-performance GPU inference'),
    ('Ray Serve', 'Python-native, pipelines, HPO-integrated', 'Complex ML workflows'),
    ('vLLM', 'LLM-specific, PagedAttention, OpenAI API', 'LLM serving at scale'),
]
print(f'{"Tool":<15} | {"Strengths":<50} | Best for')
print('-' * 100)
for tool, strengths, best in comparisons:
    print(f'{tool:<15} | {strengths:<50} | {best}')

## Summary

### Key Concepts

| Concept | API | Notes |
|---------|-----|-------|
| Define deployment | `@serve.deployment` | Class or function |
| Scale replicas | `num_replicas=N` | Auto or manual |
| Batching | `@serve.batch(max_batch_size=32)` | Group requests for throughput |
| Pipeline | `B.bind(A.bind())` | Chain deployments |
| Deploy | `serve.run(deployment.bind())` | Starts serving |
| Auto-scaling | `autoscaling_config={min: 1, max: 10, target: 2}` | Scale on demand |

**Key design principle**: separate concerns -- preprocessor deployment scales on CPU resources, model deployment scales on GPU resources. Ray Serve lets you scale each independently.

**Next**: `10_capstone_distributed_pipeline.ipynb` -- putting it all together: data -> train -> tune -> serve.
